In [1]:
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier

In [2]:
# Earth Engine Initialization and Region Definition
ee.Authenticate()
ee.Initialize(project='qsair-463811')

agricultural_region = ee.Geometry.Rectangle([32.40, -0.30, 32.70, 0.10]) 

In [3]:
# Sentinel-1 SAR Data Collection and Filtering
# Import Sentinel-1 SAR data collection.
# The "DV" (Dual Polarization) product is selected, and filtered by date and bounds.
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.eq('resolution_meters', 10)) \
    .filter(ee.Filter.date('2022-06-01', '2022-08-31')) \
    .filterBounds(agricultural_region) \
    .select(['VV', 'VH'])

# Calculate the median composite for the SAR data over the specified period.
# The median is taken for both 'VV' and 'VH' polarizations.
s1_median = s1_collection.median()

In [4]:
# Optical (Sentinel-2) Data Collection and Preprocessing
# Import Sentinel-2 Level-1C data collection.
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filter(ee.Filter.date('2022-06-01', '2022-08-31')) \
    .filterBounds(agricultural_region) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # Filters out images with more than 20% cloudy pixels

# Calculate the median composite for Sentinel-2 data.
s2_median = s2_collection.median()

# Function to add Normalized Difference Vegetation Index (NDVI) and Normalized Difference Water Index (NDWI)
def add_indices(image):
    # Calculate NDVI (Normalized Difference Vegetation Index)
    # (NIR - Red) / (NIR + Red)
    # Sentinel-2 bands: NIR = B8, Red = B4
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    # Calculate NDWI (Normalized Difference Water Index)
    # (Green - NIR) / (Green + NIR)
    # Sentinel-2 bands: Green = B3, NIR = B8
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')

    return image.addBands(ndvi).addBands(ndwi)

# Apply the function to the Sentinel-2 median image
s2_median_with_indices = add_indices(s2_median)

In [5]:
# Combining SAR and Optical Data
# Combine Sentinel-1 and Sentinel-2 data into a single image.
# Select relevant bands for combined analysis.
combined_image = s1_median.addBands(s2_median_with_indices) \
    .select(['VV', 'VH', 'B2', 'B3', 'B4', 'B8', 'NDVI', 'NDWI']) # B2:Blue, B3:Green, B4:Red, B8:NIR

# Display the combined image (example visualization parameters)
# Define visualization parameters for Sentinel-2 true color (B4, B3, B2)
# The min and max values would depend on the actual data range.
vis_params_s2 = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000, # Example max, adjust based on actual data
    'gamma': 1.4
}

# Add the combined image to the map (using geemap)
# geemap.Map() creates a new map object
m = geemap.Map(center=agricultural_region.centroid().getInfo()['coordinates'][::-1], zoom=10) # Centering map on the region
m.addLayer(combined_image, vis_params_s2, 'Combined SAR-Optical Image')
m

Map(center=[-0.09999993652857773, 32.54999999999992], controls=(WidgetControl(options=['position', 'transparen…

In [7]:
# Data Sampling for Machine Learning
training_points = ee.FeatureCollection('training-rectangle.geojson')

training_data = combined_image.sampleRegions(
    collection=training_points, # Replace 'training_points' with your actual training data FeatureCollection
    properties=['crop_type'],   # The property in your training data that holds the crop label
    scale=10 # Sample at 10-meter resolution
)

In [8]:
# Prepare Training Data for Scikit-learn
# Convert the Earth Engine FeatureCollection to a Pandas DataFrame for scikit-learn.
# Get the list of feature bands used in the combined image
feature_bands = ['VV', 'VH', 'B2', 'B3', 'B4', 'B8', 'NDVI', 'NDWI']
label = 'crop_type' # The name of the property containing the labels

# Convert to a list of dictionaries, then to a Pandas DataFrame
# Note: This step can be memory intensive for large datasets.
training_data_list = training_data.getInfo()['features']
training_features = []
training_labels = []

for feature in training_data_list:
    properties = feature['properties']
    feature_values = [properties[band] for band in feature_bands]
    training_features.append(feature_values)
    training_labels.append(properties[label])

X = np.array(training_features) # Features
y = np.array(training_labels)   # Labels

EEException: Collection.loadTable: Collection asset 'training-rectangle.geojson' not found.

In [ ]:
# Cell 8: Train the Random Forest Classifier
# Initialize and train a RandomForestClassifier.
# The number of estimators (trees) can be adjusted.
classifier = RandomForestClassifier(n_estimators=100, random_state=42) # Example: 100 trees, fixed random state for reproducibility

# Train the classifier using the prepared features (X) and labels (y)
classifier.fit(X, y)

In [ ]:
# Map the classification function over the image.
# The 'classify' method of ee.Image applies a trained classifier to each pixel.
classified_image = combined_image.select(feature_bands).classify(classifier)

In [ ]:
# Visualize the Classified Map
classification_vis_params = {
    'min': 0,
    'max': 5, # Adjust 'max' to the highest label value in your classification
    'palette': ['green', 'blue', 'yellow', 'brown', 'red', 'purple']
}

# Add the classified image to the map.
m.addLayer(classified_image, classification_vis_params, 'Crop Classification Map')

# Display the map again to show the classification
m